# 📡 TV Attribution: Spot-Level Lift Estimation

This notebook performs the granular attribution of ad airings to website traffic spikes. We identify the baseline, extract observed lift, and fit parametric response models.

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from attribution import TVAttributor
from utils.logging_config import setup_logging

logger = setup_logging("spot_attribution")
logger.info("Starting spot attribution pipeline")

# Load data
sessions = pd.read_csv('../data/sessions.csv', parse_dates=['timestamp'])
airings = pd.read_csv('../data/airings.csv', parse_dates=['timestamp'])

attributor = TVAttributor(sessions, airings)
logger.info("Attributor initialized.")

### 1. Running the Full Attribution Pipeline

We process all 300 airings, identifying which ones showed significant lift and estimating their impact parameters.

In [ ]:
results = attributor.run_attribution_pipeline()
logger.info("Attribution complete. Resulting dataframe head:")
results.head()

### 2. Identifying Missed Airings

The data includes 10 airings that were scheduled but did not result in injected lift. We evaluate our detection performance.

In [ ]:
results['detected_missed'] = results['z_score'] < 1.0
missed_summary = results[results['detected_missed'] == True]
print(f"Detected {len(missed_summary)} potential missed airings.")

### 3. Response Curve Visualization

We visualize the average fitted response curve across significant airings.

In [ ]:
sig_results = results[results['is_significant'] == True]
avg_A = sig_results['fitted_A'].mean()
avg_tau = sig_results['fitted_tau'].mean()

t_vals = np.linspace(0, 15, 100)
lift = attributor.response_model(t_vals, avg_A, avg_tau)

plt.figure(figsize=(10, 5))
plt.plot(t_vals, lift, color='red', linewidth=2)
plt.title('Average Parameteric Response Curve ($lift(t)$)')
plt.ylabel('Incremental Sessions')
plt.xlabel('Minutes Post-Airing')
plt.show()